In [1]:
## Create a vector of the required packages for this analysis
req_packages <- c("ape", "ggplot2", "ggpubr", "ggtree", "gridExtra", "janitor", "RColorBrewer",
                   "stringr", "tidyverse", "treeio", "TreeTools", "vegan", "viridis")

## load the packages, quietly
invisible(suppressWarnings(suppressMessages(
    lapply(req_packages, require, character.only = TRUE)
)))

### Determine which tree fits best

In [2]:
## load in results
null_results <- read_delim("cagee_null/results.txt", skip = 2, col_names = FALSE) %>%
    rename(assay = 1, null = 2, trash = 3, value = 4) %>%
    mutate(null = case_when(null == "Likelihood" ~ value,
                            TRUE ~ null),
           null = as.numeric(null),
           assay = case_when(assay == "Final" ~ "likelihood",
                             assay == "Sigma2:" ~ "sigma_medium")) %>%
    select(c("assay", "null")) %>%
    na.omit()

midhigh_results <- read_delim("cagee_midhigh/results.txt", skip = 2, col_names = FALSE)
colnames(midhigh_results) <- c("scrap", "sigma_high", "sigma_medium", "sigma_low")

midlow_results <- read_delim("cagee_midlow/results.txt", skip = 2, col_names = FALSE)
colnames(midlow_results) <- c("scrap", "sigma_high", "sigma_medium", "sigma_low")

midmedium_results <- read_delim("cagee_midmedium/results.txt", skip = 2, col_names = FALSE)
colnames(midmedium_results) <- c("scrap", "sigma_high", "sigma_medium", "sigma_low")

Warning message:
“One or more parsing issues, call `problems()` on your data frame for details,
e.g.:
  dat <- vroom(...)
  problems(dat)”
Rows: 4 Columns: 4
── Column specification ────────────────────────────────────────────────────────
Delimiter: " "
chr (4): X1, X2, X3, X4

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Warning message:
“There was 1 warning in `mutate()`.
ℹ In argument: `null = as.numeric(null)`.
Caused by warning:
! NAs introduced by coercion”
Warning message:
“One or more parsing issues, call `problems()` on your data frame for details,
e.g.:
  dat <- vroom(...)
  problems(dat)”
Rows: 4 Columns: 4
── Column specification ────────────────────────────────────────────────────────
Delimiter: " "
chr (4): X1, X2, X3, X4

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this

In [6]:
### clean up
mh_clean <- midhigh_results %>%
    filter(scrap %in% c("Final", "Sigma2:")) %>%
    pivot_longer(!scrap, names_to = "assay", values_to = "medium_high") %>%
    mutate(assay = case_when(scrap == "Final" ~ "likelihood",
                             TRUE ~ assay),
           medium_high = str_replace(medium_high, ",", ""),
           medium_high = as.numeric(medium_high)) %>%
    na.omit() %>%
    select(-scrap)

mm_clean <- midmedium_results %>%
    filter(scrap %in% c("Final", "Sigma2:")) %>%
    pivot_longer(!scrap, names_to = "assay", values_to = "medium_medium") %>%
    mutate(assay = case_when(scrap == "Final" ~ "likelihood",
                             TRUE ~ assay),
           medium_medium = str_replace(medium_medium, ",", ""),
           medium_medium = as.numeric(medium_medium)) %>%
    na.omit() %>%
    select(-scrap)

ml_clean <- midlow_results %>%
    filter(scrap %in% c("Final", "Sigma2:")) %>%
    pivot_longer(!scrap, names_to = "assay", values_to = "medium_low") %>%
    mutate(assay = case_when(scrap == "Final" ~ "likelihood",
                             TRUE ~ assay),
           medium_low = str_replace(medium_low, ",", ""),
           medium_low = as.numeric(medium_low)) %>%
    na.omit() %>%
    select(-scrap)

Warning message:
“There was 1 warning in `mutate()`.
ℹ In argument: `medium_high = as.numeric(medium_high)`.
Caused by warning:
! NAs introduced by coercion”


Warning message:
“There was 1 warning in `mutate()`.
ℹ In argument: `medium_medium = as.numeric(medium_medium)`.
Caused by warning:
! NAs introduced by coercion”
Warning message:
“There was 1 warning in `mutate()`.
ℹ In argument: `medium_low = as.numeric(medium_low)`.
Caused by warning:
! NAs introduced by coercion”


In [7]:
## pull the null likelihood
null_likelihood <- null_results$null[1]

## format together and get chi squared for each model
sigma_results <- mh_clean %>%
    full_join(mm_clean, by = "assay") %>%
    full_join(ml_clean, by = "assay") %>%
    t() %>%
    as.data.frame() %>%
    row_to_names(1) %>%
    mutate_all(as.numeric) %>%
    rownames_to_column("assay") %>%
    mutate(chisquare = 2*(likelihood - null_likelihood),
           pvalue = NA) %>%
    separate(assay, into = c("sperm_category_sub.mad", "node"))

## pull chi square results
chi_list <- sigma_results$chisquare

## calculate p value and populate in dataframe
for (i in 1:length(chi_list)){
    result <- pchisq(q = chi_list[[i]], df = 2, lower.tail = FALSE)
    sigma_results$pvalue[[i]] <- result
}

sigma_results

sperm_category_sub.mad,node,likelihood,sigma_high,chisquare,pvalue
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>
medium,high,553504,0.806346,24722,0
medium,medium,553219,0.787360,24152,0
medium,low,553378,0.776594,24470,0


### Rerun gene expression analysis with new model

In [8]:
## load in data
change <- read.delim("cagee_midlow/change.tab") %>%
    janitor::clean_names()

In [9]:
colnames(change)

[1] "transcript_id"              "drosophila_bifasciata_1"   
 [3] "drosophila_obscura_2"       "drosophila_subobscura_3"   
 [5] "drosophila_madeirensis_4"   "drosophila_guanche_5"      
 [7] "drosophila_azteca_6"        "drosophila_affinis_7"      
 [9] "drosophila_pseudoobscura_8" "x_9"                       
[11] "x_10"                       "x_11"                      
[13] "x_12"                       "x_13"                      
[15] "x_14"                       "x_15"

In [10]:

## get list of branches with increases
increase_branches <- c("x_12", "x_14", "x_15")

## determine how many genes are significant at each branch
credible_genes <- change %>%
    select(transcript_id, any_of(increase_branches)) %>%
    clean_names() %>%
    filter(grepl("\\*", x_12) | grepl("\\*", x_14) | grepl("\\*", x_15)) %>%
    mutate(signif = case_when(grepl("\\*", x_12) & grepl("\\*", x_14) & grepl("\\*", x_15) ~ "12,14,15",
                              grepl("\\*", x_12) & grepl("\\*", x_14) ~ "12,14", 
                              grepl("\\*", x_12) & grepl("\\*", x_15) ~ "12,15",
                              grepl("\\*", x_14) & grepl("\\*", x_15) ~ "14,15", 
                              grepl("\\*", x_12) ~ "12",
                              grepl("\\*", x_14) ~ "14",
                              grepl("\\*", x_15) ~ "15"))

## count number of shared genes
credible_genes %>%
    count(signif)

signif,n
<chr>,<int>
12,10
"12,15",2
14,7
15,544


In [11]:
## write the results to a table
write_csv(credible_genes, "credible_genes.csv")